# 01 — Stiefel Data + Diffusion Training

This notebook generates Stiefel data and trains a small score model for a quick reproducible smoke run.

### Setup

Install dependencies, set random seeds, and select device.

In [ ]:
from pathlib import Path
import os
import random
import subprocess
import sys

import numpy as np
import torch

# If the repo is not present in /content, set REPO_URL to your GitHub repo and rerun.
REPO_URL = "https://github.com/<your-org>/score-manifold-optimization.git"
REPO_DIR = Path("/content/score-manifold-optimization")

if not (REPO_DIR / "pyproject.toml").exists():
    if "<" in REPO_URL:
        raise RuntimeError(
            "Repo not found at /content/score-manifold-optimization. "
            "Please set REPO_URL to your repository URL or clone manually."
        )
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Repo: {REPO_DIR}")
print(f"Device: {DEVICE}")

### Local Setup (Alternative)

If the repo is already cloned locally, run this cell instead of the Colab setup cell above.

In [ ]:
from pathlib import Path
import os
import random

import numpy as np
import torch

# Local alternative: use this when the repository is already cloned.
start = Path.cwd().resolve()
REPO_DIR = next((p for p in [start, *start.parents] if (p / "pyproject.toml").exists()), None)
if REPO_DIR is None:
    raise RuntimeError("Could not find repo root (missing pyproject.toml in parent dirs).")

os.chdir(REPO_DIR)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Repo root: {REPO_DIR}")
print(f"Device: {DEVICE}")

# Install once from terminal (repo root): pip install -e .


### Generate Data

Create and save a Stiefel(3,3) manifold dataset.

In [ ]:
from datetime import datetime
from diffusion.spaces import Stiefel

DATA_DIR = Path(f"{REPO_DIR}/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH = DATA_DIR / "stiefel_n3_p3.pt"

constraint = Stiefel(n=3, p=3)
space = constraint.get_space()

train_data = constraint.sample(500)
test_data = constraint.sample(100)
val_data = constraint.sample(100)

metadata = {
    "space": {
        "class": "MatrixSpace",
        "params": {"m": int(space.m), "n": int(space.n)},
    },
    "constraint": {
        "class": "Stiefel",
        "params": {"n": int(constraint.n), "p": int(constraint.p)},
    },
    "n_train": int(train_data.shape[0]),
    "n_test": int(test_data.shape[0]),
    "n_val": int(val_data.shape[0]),
    "created": datetime.now().isoformat(),
    "version": "1.0",
    "generation_config": {
        "generator_type": "manifold",
        "noise_std": 0.0,
    },
}

dataset = {
    "train_data": train_data,
    "test_data": test_data,
    "val_data": val_data,
    "metadata": metadata,
}
torch.save(dataset, DATASET_PATH)

print(f"Train shape: {tuple(train_data.shape)}")
print(f"Test shape:  {tuple(test_data.shape)}")
print(f"Val shape:   {tuple(val_data.shape)}")
print(f"Saved dataset: {DATASET_PATH}")


### Train Model

Run diffusion-train and save artifacts to a fixed output directory.

In [ ]:
from datetime import datetime
import json
import yaml

from diffusion.data import load_dataset
from diffusion.models import create_model
from diffusion.training import DiffusionTrainer, TrainingOptions, create_diffusion, create_optimizer

CONFIG_PATH = REPO_DIR / "configs/train_stiefel_quick.yaml"
OUTPUT_DIR = REPO_DIR / "outputs/stiefel_train_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

if not isinstance(cfg.get("dataset"), dict):
    cfg["dataset"] = {}
cfg["dataset"]["path"] = str(DATASET_PATH)
dataset = load_dataset(str(DATASET_PATH), device=DEVICE)
space = dataset.get_space()
diffusion = create_diffusion(space, cfg)
model = create_model(
    model_type=str(cfg["model"]["type"]),
    space=space,
    diffusion=diffusion,
    config=cfg,
    initialize=True,
).to(DEVICE)
optimizer = create_optimizer(model, cfg)

options_dict = dict(cfg.get("training_options", {}))
options_dict["batch_size"] = int(cfg.get("training", {}).get("batch_size", 32))
options = TrainingOptions(**options_dict)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
constraint = dataset.constraint

def log_fn(samples):
    violations = constraint.violation(samples)
    return {
        "constraint_violation_mean": violations.mean().item(),
        "constraint_violation_std": violations.std().item(),
        "constraint_violation_max": violations.max().item(),
    }

trainer = DiffusionTrainer(
    diffusion=diffusion,
    score_model=model,
    train_data=dataset.train_data.to(DEVICE),
    optimizer=optimizer,
    options=options,
    log_fn=log_fn,
    device=DEVICE,
    output_dir=OUTPUT_DIR,
    metadata=dataset.metadata,
)

num_epochs = int(cfg.get("training", {}).get("num_epochs", 100))
trainer.train(num_epochs=num_epochs)
trainer.save_checkpoint(str(OUTPUT_DIR / "checkpoint.pt"))

with open(OUTPUT_DIR / "config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

summary = {
    "timestamp": datetime.now().isoformat(),
    "output_dir": str(OUTPUT_DIR),
    "num_epochs": num_epochs,
    "final_loss": float(trainer.train_losses[-1]) if trainer.train_losses else None,
    "num_params": num_params,
    "dataset_path": str(DATASET_PATH),
    "model_type": str(cfg["model"]["type"]),
}
with open(OUTPUT_DIR / "train_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Training complete. Artifacts written to {OUTPUT_DIR}")

### Verify Outputs

Check required checkpoint files and print train summary metrics.

In [ ]:
import json

required = ["model.pth", "checkpoint_data.pt", "config.yaml", "metadata.json", "train_summary.json"]
for name in required:
    path = OUTPUT_DIR / name
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")

summary_path = OUTPUT_DIR / "train_summary.json"
with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

print("\
Train summary:")
print(json.dumps(summary, indent=2))

### Sample from pretrained model

Generate samples from the trained model and measure distance to the manifold.

In [ ]:
import yaml
import torch
from diffusion.models import create_model

# Load model
with open(OUTPUT_DIR / "config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

model = create_model(
    model_type=str(cfg["model"]["type"]),
    space=space,
    diffusion=diffusion,
    config=cfg,
    initialize=False,
).to(DEVICE)
state_dict = torch.load(OUTPUT_DIR / "model.pth", map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

#Or load from context:
#context = load_pretrained_score_context(
#    checkpoint_dir=OUTPUT_DIR,
#    dataset_path_override=DATASET_PATH,
#    device=DEVICE,
#)
#model = context.model
#diffusion = context.diffusion
#constraint = context.constraint

N_SAMPLES = 128
with torch.no_grad():
    samples = diffusion.sample_reverse(
        N_sample=N_SAMPLES,
        score_model=model,
        flow_type="ODE",
        end_only=True,
        device=DEVICE,
    )

dist = constraint.distance(samples).detach().cpu()

print(f"Samples shape: {tuple(samples.shape)}")
print(f"Manifold distance | mean={dist.mean().item():.4e}, "
      f"median={dist.median().item():.4e}, max={dist.max().item():.4e}")